# PAUL Open Model — Autonomous Baseline Evaluation (Gemma 4 E4B IT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/foundrypaul-cloud/paul-open/blob/main/notebooks/03_baseline_evaluation_e4b.ipynb)

This notebook executes the official **Phase 2 Autonomous Baseline Evaluation** for the **PAUL Open Model** research project in Google Colab (Tesla T4 free runtime, ~14.56 GiB usable VRAM).

### Fully Autonomous Execution & Persistence Architecture
Once you start the Colab runtime and click **Runtime $\rightarrow$ Run all**, this notebook autonomously:
1. Audits the GPU and installs pinned dependencies.
2. Authenticates securely via `HF_TOKEN` from Google Colab Secrets (zero token leakage).
3. Automatically detects whether Google Drive is mounted (optional persistent mirroring).
4. Loads `google/gemma-4-E4B-it` in 4-bit NF4 double-quantization mode (`compute_dtype=torch.float16`).
5. Evaluates all 50 baseline cases across 10 capability domains and 10 Indian/global languages (`BASELINE_VERSION = 1.0.0`).
6. Checkpoints progress incrementally to disk (`checkpoint.jsonl`), enabling seamless resume.
7. Exports structured artifacts under `results/baseline/<experiment_id>/` (`manifest.json`, `results.json`, `results.csv`, `summary.md`, `metadata.json`, `STATUS.json`, `execution.log`).
8. Performs GPU memory reclamation and emits final integrity verification.

> **Safety Boundaries**: Zero fine-tuning, zero dataset downloads, zero weight modifications, zero LLM judge, and zero credential leakage.

## Step 1: Autonomous Repository Setup & Package Installation
Clones the repository if running inside Colab and installs the pinned Gemma 4 software stack.

In [ ]:
import os
import sys

# Clone repository if running in Colab
if "google.colab" in sys.modules or os.environ.get("COLAB_GPU") is not None:
    if not os.path.exists("paul-open"):
        !git clone -q https://github.com/foundrypaul-cloud/paul-open.git
        %cd paul-open
    elif os.path.basename(os.getcwd()) != "paul-open":
        %cd paul-open

# Install explicit, pinned versions of the Gemma 4 ML stack
!pip install -q --no-cache-dir \
    "torch>=2.11.0" \
    "transformers>=5.13.1" \
    "peft>=0.19.0" \
    "bitsandbytes>=0.45.0" \
    "accelerate>=1.2.0" \
    "datasets>=3.2.0" \
    "huggingface_hub>=0.28.0" \
    "pyyaml>=6.0" \
    "rich>=13.0.0" \
    "sentencepiece>=0.2.0" \
    "tokenizers>=0.21.0" \
    "pandas>=2.2.0"

# Configure src path
if os.path.exists("src") and "src" not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))

## Step 2: Hardware Preflight & Dependency Verification
Checks GPU name, total VRAM, CUDA runtime, PyTorch, Transformers, PEFT, and BitsAndBytes.

In [ ]:
import platform
import bitsandbytes as bnb
import peft
import torch
import transformers

print("=" * 70)
print(" AUTONOMOUS HARDWARE PREFLIGHT")
print("=" * 70)
print(f" Python Version      : {platform.python_version()}")
print(f" PyTorch Version     : {torch.__version__}")
print(f" Transformers Version: {transformers.__version__} (>=5.10.1 required for Gemma 4)")
print(f" PEFT Version        : {peft.__version__} (>=0.19.0 required for Gemma 4 QLoRA)")
print(f" BitsAndBytes Version: {bnb.__version__}")
print("-" * 70)

if not torch.cuda.is_available():
    raise SystemError("CRITICAL: No CUDA GPU detected! Please select a T4/L4 GPU runtime in Colab.")

gpu_props = torch.cuda.get_device_properties(0)
gpu_name = gpu_props.name
total_vram_gb = gpu_props.total_memory / (1024 ** 3)
print(f" GPU Device          : {gpu_name}")
print(f" Total VRAM          : {total_vram_gb:.2f} GiB")
print(f" CUDA Version        : {torch.version.cuda}")
print("✓ Hardware preflight checks PASSED.")
print("=" * 70)

## Step 3: Secure Hugging Face Authentication
Authenticates with `HF_TOKEN` from Google Colab Secrets (zero token leakage) and verifies access to `google/gemma-4-E4B-it`.

In [ ]:
import os
from huggingface_hub import HfApi, login

MODEL_ID = "google/gemma-4-E4B-it"

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "MISSING HF_TOKEN: Please configure 'HF_TOKEN' in Colab Secrets (Key icon on left sidebar)."
    )

login(token=hf_token, add_to_git_credential=False)
api = HfApi()
try:
    model_info = api.model_info(MODEL_ID, token=hf_token)
    print(f"✓ Verified access to gated model: {MODEL_ID}")
except Exception as e:
    raise PermissionError(
        f"ACCESS DENIED to {MODEL_ID}. Please accept the Gemma license terms at https://huggingface.co/{MODEL_ID}"
    ) from e

## Step 4: 4-Bit Model Loading & Quantization Setup
Loads `google/gemma-4-E4B-it` in the validated 4-bit NF4 double-quantization configuration.

In [ ]:
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Configuring 4-bit NF4 Quantization (compute_dtype: {compute_dtype})...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, token=hf_token)

print(f"Loading {MODEL_ID} weights via AutoModelForMultimodalLM in 4-bit NF4...")
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    token=hf_token,
)

allocated_gb = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"✓ Model loaded successfully. Initial VRAM Allocated: {allocated_gb:.2f} GiB.")

## Step 5: Check Persistence Mode & Run Autonomous Evaluation
Detects Google Drive mount status (non-blocking), initializes the versioned 50-case benchmark suite (`BASELINE_VERSION = 1.0.0`), and runs the `EvaluationRunner` with incremental checkpointing (`checkpoint.jsonl`).

In [ ]:
import os
from pathlib import Path
from datetime import datetime, UTC
from paul_open_model.evaluation import (
    BASELINE_VERSION,
    EvaluationRunner,
    get_baseline_benchmark_suite,
)

# Check for optional Google Drive mount without blocking authentication
DRIVE_MOUNT_POINT = Path("/content/drive/MyDrive")
DRIVE_BASE_DIR = DRIVE_MOUNT_POINT / "paul-open-experiments" / "baseline"

if DRIVE_MOUNT_POINT.exists():
    print(f"✓ Google Drive detected. Persistence Mode: DRIVE_MIRRORED ({DRIVE_BASE_DIR})")
    drive_path = DRIVE_BASE_DIR
else:
    print("ℹ Google Drive is NOT mounted. Persistence Mode: RUNTIME_LOCAL (ephemeral storage).")
    print("  (Checkpoints will persist within this runtime session only.)")
    drive_path = None

# Generate unique experiment ID
now_tag = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
EXPERIMENT_ID = f"exp_gemma4_e4b_baseline_{now_tag}"

suite = get_baseline_benchmark_suite()
print(f"Loaded PAUL Open Model Baseline Benchmark Suite (v{suite.version}) with {len(suite)} cases.")

runner = EvaluationRunner(
    model=model,
    processor=processor,
    suite=suite,
    model_id=MODEL_ID,
    experiment_id=EXPERIMENT_ID,
    output_dir="results/baseline",
    drive_backup_dir=drive_path,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    resume=True,
    gpu_device=gpu_name,
    total_vram_gb=total_vram_gb,
)

print(f"Starting autonomous baseline evaluation: {EXPERIMENT_ID}...")
results = runner.run_all(verbose=True)

## Step 6: Display Manifest & Scorecard Reports
Displays the execution manifest, domain scorecard, and language breakdown.

In [ ]:
import pandas as pd

exp_dir = runner.exp_dir
manifest_dict = runner.generate_manifest_dict()
status_dict = runner.generate_status_dict()

print("=" * 70)
print(" EXECUTION MANIFEST & ARTIFACTS")
print("=" * 70)
for k, v in manifest_dict.items():
    print(f" {k:<25}: {v}")
print("=" * 70)

# Display summary DataFrame
df = pd.read_csv(exp_dir / "results.csv")
print("\n--- Sample Results (First 5 Cases) ---")
display_cols = ["case_id", "domain", "language", "heuristic_rubric_score", "latency_seconds", "human_review_required"]
print(df[display_cols].head(5).to_string(index=False))

print("\n--- Domain Scorecard ---")
domain_summary = df.groupby("domain").agg({
    "case_id": "count",
    "heuristic_rubric_score": "mean",
    "latency_seconds": "mean",
    "human_review_required": "sum",
}).rename(columns={
    "case_id": "Cases",
    "heuristic_rubric_score": "Mean Rubric (0-100)",
    "latency_seconds": "Mean Latency (s)",
    "human_review_required": "Human Review Cases",
})
print(domain_summary.round(2).to_string())

print("\n--- Language Breakdown ---")
lang_summary = df.groupby("language").agg({
    "case_id": "count",
    "heuristic_rubric_score": "mean",
    "latency_seconds": "mean",
}).rename(columns={
    "case_id": "Cases",
    "heuristic_rubric_score": "Mean Rubric (0-100)",
    "latency_seconds": "Mean Latency (s)",
})
print(lang_summary.round(2).to_string())

## Step 7: GPU Memory Cleanup & Final Verification
Reclaims GPU memory tensors, audits secret exclusion, and verifies execution status.

In [ ]:
import gc

peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0

# Cleanup model and tensors
del model
del processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

remaining_vram_gb = torch.cuda.memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0

status_dict = runner.generate_status_dict()
all_cases_passed = status_dict["status"] == "SUCCESS"

print("=" * 70)
print(" FINAL BASELINE EXECUTION SUMMARY")
print("=" * 70)
for k, v in status_dict.items():
    print(f" {k:<30}: {v}")
print(f" {'peak_vram_observed_gb':<30}: {peak_vram_gb:.2f} GiB")
print(f" {'vram_retained_after_cleanup':<30}: {remaining_vram_gb:.2f} GiB")
print(f" {'training_occurred':<30}: False")
print(f" {'dataset_downloads':<30}: False")
print("=" * 70)

if all_cases_passed:
    print("✓ PHASE 2 BASELINE BENCHMARK FULLY COMPLETED (50/50 cases evaluated).")
    print(f"  Local results  : {runner.exp_dir}")
    if runner.drive_exp_dir is not None:
        print(f"  Drive mirror   : {runner.drive_exp_dir}")
else:
    print(f"⚠ STATUS: {status_dict['status']} ({status_dict['completed_cases']}/50 cases completed).")